In [4]:
"""Benchmark: Jacobian of steady-state loss function.

Compares dense vs GMRES+precond at various krylov sizes, for forward + Jacobian.
Also compares vmap vs lax.map batching.

No catographer dependency. N=30 to keep runtime reasonable.
"""

import gc
import time

import dynamiqs as dq
import jax
import jax.numpy as jnp
import equinox as eqx

dq.set_matmul_precision("highest")
dq.set_precision("double")

N = 40
a = dq.destroy(N)
n_hat_jax = dq.number(N).to_jax()
n = N
twopi = 2 * jnp.pi

# Precomputed operator matrices
a_jax = a.to_jax()
adag_jax = a.dag().to_jax()
adag2a2 = (a.dag() @ a.dag() @ a @ a).to_jax()
adaga = (a.dag() @ a).to_jax()
I_vec = jnp.eye(n, dtype=jnp.complex128).flatten(order="F")

n_detunings = 15
delta_vals = jnp.linspace(-20, 20, n_detunings) * twopi

# Fit parameters: [kappa, kerr, eps]
params_true = jnp.array([14.0 * twopi, -1.0 * twopi, 16.0])


def build_H_and_Ls(params, delta):
    kap, kerr, ep = params
    H = (
        -kerr / 2 * adag2a2
        - delta * adaga
        + 1j * jnp.sqrt(kap) * ep * a_jax
        - 1j * jnp.sqrt(kap) * ep * adag_jax
    )
    L = jnp.sqrt(kap) * a_jax
    return dq.asqarray(H), [dq.asqarray(L)]


# Dense: build superoperator, direct solve
def dense_single(params, delta):
    H_q, Ls_q = build_H_and_Ls(params, delta)
    L_sup = dq.slindbladian(H_q, Ls_q).to_jax()
    L_def = L_sup + jnp.outer(I_vec, I_vec)
    x = jnp.linalg.solve(L_def, I_vec)
    rho = x.reshape((n, n), order="F")
    rho = (rho + rho.conj().T) / 2
    rho /= jnp.trace(rho)
    return jnp.trace(rho @ n_hat_jax).real


# GMRES with Lyapunov preconditioner (dynamiqs)
def gmres_single(params, delta, ks=64):
    H_q, Ls_q = build_H_and_Ls(params, delta)
    solver = dq.SteadyStateGMRES(krylov_size=ks, tol=1e-6, max_iteration=100)
    result = dq.steadystate(H_q, Ls_q, solver=solver)
    return jnp.trace(result.rho.to_jax() @ n_hat_jax).real


# ── Compute dense reference ──────────────────────────────────────
print(f"Kerr oscillator: N={N}, {n_detunings} detunings, 3 fit params")
print("Jacobian: jacfwd of <n>(params) w.r.t. [κ, K, ε]")
print("Computing dense reference...")
ref_fwd_fn = eqx.filter_jit(lambda p: jax.lax.map(lambda d: dense_single(p, d), delta_vals))
n_ref = ref_fwd_fn(params_true).block_until_ready()
ref_jac_fn = eqx.filter_jit(
    jax.jacfwd(lambda p: jax.lax.map(lambda d: dense_single(p, d), delta_vals))
)
jac_ref = ref_jac_fn(params_true).block_until_ready()
print("Done.\n")


def run_benchmark(label, single_fn, krylov_sizes):
    print(f"\n{'=' * 95}")
    print(f"{label}")
    print(f"{'=' * 95}")
    print(
        f"  {'ks':>4}  {'mode':>8}  {'fwd(ms)':>8}  {'jac(ms)':>8}  "
        f"{'total(ms)':>10}  {'fwd_err':>10}  {'jac_err':>10}"
    )
    print(f"  {'-' * 82}")

    for ks in krylov_sizes:
        if ks is None:
            solve_fn = single_fn
        else:
            solve_fn = lambda p, d, _ks=ks: single_fn(p, d, _ks)

        for mode in ["vmap", "lax.map"]:
            jax.clear_caches()
            gc.collect()

            if mode == "vmap":
                batched = lambda p: jax.vmap(solve_fn, in_axes=(None, 0))(p, delta_vals)
            else:
                batched = lambda p: jax.lax.map(lambda d: solve_fn(p, d), delta_vals)

            # Forward
            fn_fwd = eqx.filter_jit(batched)
            _ = fn_fwd(params_true).block_until_ready()
            ts = []
            for _ in range(3):
                t0 = time.time()
                vals = fn_fwd(params_true).block_until_ready()
                ts.append(time.time() - t0)
            t_fwd = min(ts) * 1000
            fwd_err = float(jnp.max(jnp.abs(vals - n_ref)))

            # Jacobian
            fn_jac = eqx.filter_jit(jax.jacfwd(batched))
            _ = fn_jac(params_true).block_until_ready()
            ts = []
            for _ in range(3):
                t0 = time.time()
                jac = fn_jac(params_true).block_until_ready()
                ts.append(time.time() - t0)
            t_jac = min(ts) * 1000
            jac_err = float(jnp.max(jnp.abs(jac - jac_ref)))
            has_nan = bool(jnp.any(jnp.isnan(jac)))

            ks_str = f"{ks}" if ks is not None else "  —"
            nan_tag = " NaN!" if has_nan else ""
            wrong_tag = " ←WRONG" if fwd_err > 0.1 else ""
            print(
                f"  {ks_str:>4}  {mode:>8}  {t_fwd:8.0f}  {t_jac:8.0f}  "
                f"{t_fwd + t_jac:10.0f}  {fwd_err:10.2e}{wrong_tag}  "
                f"{jac_err:10.2e}{nan_tag}"
            )


# ── Run benchmarks ───────────────────────────────────────────────
run_benchmark("DENSE SOLVER", dense_single, [None])

run_benchmark(
    "GMRES",
    gmres_single,
    [96],
)

Kerr oscillator: N=40, 15 detunings, 3 fit params
Jacobian: jacfwd of <n>(params) w.r.t. [κ, K, ε]
Computing dense reference...
Done.


DENSE SOLVER
    ks      mode   fwd(ms)   jac(ms)   total(ms)     fwd_err     jac_err
  ----------------------------------------------------------------------------------
     —      vmap      2884      3984        6868    0.00e+00    0.00e+00
     —   lax.map      2648      2848        5495    0.00e+00    0.00e+00

GMRES
    ks      mode   fwd(ms)   jac(ms)   total(ms)     fwd_err     jac_err
  ----------------------------------------------------------------------------------
    96      vmap    124913    395475      520387    4.32e-03    9.37e-04


EquinoxRuntimeError: Above is the stack outside of JIT. Below is the stack inside of JIT:
  File "/var/folders/f5/_r903g3n25j7j7j99lqrrrg40000gn/T/ipykernel_91042/1423751001.py", line 108, in <lambda>
    batched = lambda p: jax.lax.map(lambda d: solve_fn(p, d), delta_vals)
                        ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/pgregory/Documents/dynamiqs/.venv/lib/python3.13/site-packages/equinox/internal/_primitive.py", line 180, in _wrapper
    primals_out, tangents_out = rule(primals, tangents)
                                ~~~~^^^^^^^^^^^^^^^^^^^
  File "/Users/pgregory/Documents/dynamiqs/.venv/lib/python3.13/site-packages/lineax/_solve.py", line 244, in _linear_solve_jvp
    sol, _, _ = eqxi.filter_primitive_bind(
                ~~~~~~~~~~~~~~~~~~~~~~~~~~^
        linear_solve_p, operator, state, vecs, options, solver, True
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/Users/pgregory/Documents/dynami

In [3]:
"""
Benchmark: Jacobian of steady-state loss function (sequential solves).

- Computes dense reference (batched) once (printed as 1 summary line).
- Then runs GMRES solvers per detuning with detailed per-line output:
    delta | fwd | fwd_err | jacfwd | jac_err | max|L(rho)|

No cartographer dependency. N=40.
"""

import gc
import time

import dynamiqs as dq
import jax
import jax.numpy as jnp

# -----------------------------------------------------------------------------
# Global settings
# -----------------------------------------------------------------------------
dq.set_matmul_precision("highest")
dq.set_precision("double")

# -----------------------------------------------------------------------------
# Problem setup
# -----------------------------------------------------------------------------
N = 40
n = N
twopi = 2 * jnp.pi

a = dq.destroy(N)
n_hat_jax = dq.number(N).to_jax()

a_jax = a.to_jax()
adag_jax = a.dag().to_jax()
adag2a2 = (a.dag() @ a.dag() @ a @ a).to_jax()
adaga = (a.dag() @ a).to_jax()

I_vec = jnp.eye(n, dtype=jnp.complex128).flatten(order="F")

n_detunings = 15
delta_vals = jnp.linspace(-20, 20, n_detunings) * twopi

# Fit parameters: [kappa, kerr, eps]
params_true = jnp.array([14.0 * twopi, -1.0 * twopi, 16.0])


def build_H_and_Ls(params, delta):
    kap, kerr, ep = params
    H = (
        -kerr / 2 * adag2a2
        - delta * adaga
        + 1j * jnp.sqrt(kap) * ep * a_jax
        - 1j * jnp.sqrt(kap) * ep * adag_jax
    )
    L = jnp.sqrt(kap) * a_jax
    return dq.asqarray(H), [dq.asqarray(L)]


# -----------------------------------------------------------------------------
# Solvers
# -----------------------------------------------------------------------------
def dense_single_with_rho(params, delta):
    H_q, Ls_q = build_H_and_Ls(params, delta)

    L_sup = dq.slindbladian(H_q, Ls_q).to_jax()
    L_def = L_sup + jnp.outer(I_vec, I_vec)

    x = jnp.linalg.solve(L_def, I_vec)
    rho = x.reshape((n, n), order="F")

    rho = (rho + rho.conj().T) / 2
    rho /= jnp.trace(rho)

    nval = jnp.trace(rho @ n_hat_jax).real
    return nval, rho, H_q, Ls_q


def gmres_single_with_rho(params, delta, ks=64):
    H_q, Ls_q = build_H_and_Ls(params, delta)
    solver = dq.SteadyStateGMRES(
        krylov_size=ks,
        tol=1e-6,
        max_iteration=1000,
    )
    result = dq.steadystate(H_q, Ls_q, solver=solver)

    rho = result.rho.to_jax()
    nval = jnp.trace(rho @ n_hat_jax).real
    return nval, rho, H_q, Ls_q


# -----------------------------------------------------------------------------
# Dense reference (batched) — computed once, printed as 1 summary line
# -----------------------------------------------------------------------------
print(f"Kerr oscillator: N={N}, {n_detunings} detunings, 3 fit params")
print("Jacobian: jacfwd of <n>(params) w.r.t. [κ, K, ε]")
print("Computing dense reference (batched)...\n")

ref_fwd_fn = jax.jit(
    lambda p: jax.lax.map(lambda d: dense_single_with_rho(p, d)[0], delta_vals)
)
n_ref = ref_fwd_fn(params_true).block_until_ready()

ref_jac_fn = jax.jit(
    jax.jacfwd(lambda p: jax.lax.map(lambda d: dense_single_with_rho(p, d)[0], delta_vals))
)
jac_ref = ref_jac_fn(params_true).block_until_ready()

# Compute max Lindbladian norm across all detunings for the dense solver
dense_lind_norms = []
fwd_tuple_dense = jax.jit(lambda p, d: dense_single_with_rho(p, d))
for d in list(delta_vals):
    _, rho, H_q, Ls_q = fwd_tuple_dense(params_true, d)
    Lrho = dq.lindbladian(H_q, Ls_q, dq.asqarray(rho)).to_jax()
    dense_lind_norms.append(float(jnp.max(jnp.abs(Lrho))))

print(f"{'='*80}")
print(f"DENSE REFERENCE (summary)")
print(f"{'='*80}")
print(
    f"  <n> range: [{float(jnp.min(n_ref)):.6e}, {float(jnp.max(n_ref)):.6e}]  "
    f"max|L(rho)|: {max(dense_lind_norms):.2e}"
)
print(f"{'='*80}")


# -----------------------------------------------------------------------------
# GMRES: detailed per-detuning output
# -----------------------------------------------------------------------------
def print_gmres_per_system(ks):
    label = f"GMRES(ks={ks})"
    print(f"\n{'='*150}")
    print(f"{label} — per detuning: delta | fwd | fwd_err | jacfwd | jac_err | max|L(rho)|")
    print(f"{'='*150}")
    print(
        f"{'delta(rad/s)':>14}  {'fwd(<n>)':>12}  {'fwd_err':>10}  "
        f"{'jacfwd[dκ,dK,dε]':>44}  {'jac_maxerr':>10}  {'max|L(rho)|':>14}"
    )
    print(f"{'-'*150}")

    solve = lambda p, d: gmres_single_with_rho(p, d, ks)

    fwd_tuple = jax.jit(lambda p, d: solve(p, d))
    jac_one = jax.jit(jax.jacfwd(lambda p, d: solve(p, d)[0]))

    # warmups
    n0, rho0, _, _ = fwd_tuple(params_true, delta_vals[0])
    n0.block_until_ready()
    _ = jac_one(params_true, delta_vals[0]).block_until_ready()

    for i, d in enumerate(list(delta_vals)):
        nval, rho, H_q, Ls_q = fwd_tuple(params_true, d)
        grad = jac_one(params_true, d)

        nval.block_until_ready()
        grad.block_until_ready()

        fwd_err = float(jnp.abs(nval - n_ref[i]))
        jac_err = float(jnp.max(jnp.abs(grad - jac_ref[i])))

        Lrho = dq.lindbladian(H_q, Ls_q, dq.asqarray(rho)).to_jax()
        lind_norm = jnp.max(jnp.abs(Lrho))
        lind_norm.block_until_ready()

        print(
            f"{float(d):14.6e}  "
            f"{float(nval):12.6e}  "
            f"{fwd_err:10.2e}  "
            f"[{float(grad[0]): .3e}, {float(grad[1]): .3e}, {float(grad[2]): .3e}]  "
            f"{jac_err:10.2e}  "
            f"{float(lind_norm):14.6e}"
        )


# -----------------------------------------------------------------------------
# Sequential benchmark runner (timing + aggregate errors)
# -----------------------------------------------------------------------------
def run_benchmark_sequential(label, single_with_rho_fn, krylov_sizes):
    print(f"\n{'=' * 95}")
    print(f"{label} (sequential per-detuning)")
    print(f"{'=' * 95}")
    print(
        f"  {'ks':>4}  {'fwd_total(ms)':>13}  {'jac_total(ms)':>13}  "
        f"{'total(ms)':>10}  {'fwd_err':>10}  {'jac_err':>10}"
    )
    print(f"  {'-' * 82}")

    for ks in krylov_sizes:
        if ks is None:
            solve = lambda p, d: single_with_rho_fn(p, d)
        else:
            solve = lambda p, d: single_with_rho_fn(p, d, ks)

        fwd_one = jax.jit(lambda p, d: solve(p, d)[0])
        jac_one = jax.jit(jax.jacfwd(lambda p, d: solve(p, d)[0]))

        jax.clear_caches()
        gc.collect()

        # ---- Forward timing ----
        _ = fwd_one(params_true, delta_vals[0]).block_until_ready()
        t0 = time.time()
        vals = []
        for d in list(delta_vals):
            vals.append(fwd_one(params_true, d).block_until_ready())
        t_fwd = (time.time() - t0) * 1000.0
        vals = jnp.stack(vals)
        fwd_err = float(jnp.max(jnp.abs(vals - n_ref)))

        # ---- Jacobian timing ----
        _ = jac_one(params_true, delta_vals[0]).block_until_ready()
        t0 = time.time()
        jacs = []
        for d in list(delta_vals):
            jacs.append(jac_one(params_true, d).block_until_ready())
        t_jac = (time.time() - t0) * 1000.0
        jac = jnp.stack(jacs)

        jac_err = float(jnp.max(jnp.abs(jac - jac_ref)))
        has_nan = bool(jnp.any(jnp.isnan(jac)))

        ks_str = f"{ks}" if ks is not None else "  —"
        nan_tag = " NaN!" if has_nan else ""
        wrong_tag = " ←WRONG" if fwd_err > 0.1 else ""
        print(
            f"  {ks_str:>4}  {t_fwd:13.0f}  {t_jac:13.0f}  "
            f"{t_fwd + t_jac:10.0f}  {fwd_err:10.2e}{wrong_tag}  "
            f"{jac_err:10.2e}{nan_tag}"
        )


# -----------------------------------------------------------------------------
# Run
# -----------------------------------------------------------------------------

for ks in [96]:
    print_gmres_per_system(ks)

Kerr oscillator: N=40, 15 detunings, 3 fit params
Jacobian: jacfwd of <n>(params) w.r.t. [κ, K, ε]
Computing dense reference (batched)...

DENSE REFERENCE (summary)
  <n> range: [1.152176e+00, 1.038948e+01]  max|L(rho)|: 1.33e-13

GMRES(ks=96) — per detuning: delta | fwd | fwd_err | jacfwd | jac_err | max|L(rho)|
  delta(rad/s)      fwd(<n>)     fwd_err                              jacfwd[dκ,dK,dε]  jac_maxerr     max|L(rho)|
------------------------------------------------------------------------------------------------------------------------------------------------------
 -1.256637e+02  1.152176e+00    9.33e-15  [ 9.578e-03,  1.560e-02,  1.314e-01]    1.11e-15    3.895855e-14
 -1.077117e+02  1.450849e+00    1.22e-14  [ 1.091e-02,  2.641e-02,  1.600e-01]    1.14e-15    1.742822e-13
 -8.975979e+01  1.853911e+00    1.33e-15  [ 1.204e-02,  4.550e-02,  1.949e-01]    3.05e-15    1.044431e-13
 -7.180783e+01  2.395747e+00    8.88e-16  [ 1.252e-02,  7.866e-02,  2.357e-01]    1.14e-15    9.55